In [1]:
import pandas as pd
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import zipfile
import os
import glob

os.chdir("/Users/adnan/Desktop/us-flight-delays")

In [2]:
columns_needed = [
    'FlightDate',
    'Reporting_Airline',
    'Origin',
    'OriginStateName',
    'Dest',
    'DestStateName',
    'DepDelay',
    'ArrDelay',
    'Cancelled',
    'Diverted',
    'CarrierDelay',
    'WeatherDelay',
    'NASDelay',
    'SecurityDelay',
    'LateAircraftDelay',
    'Distance',
    'AirTime',
    'CRSDepTime'
]

zip_files = sorted(glob.glob("data/raw/*.zip"))
print(f"Found {len(zip_files)} zip files")
print(f"First: {zip_files[0]}")
print(f"Last: {zip_files[-1]}")

with zipfile.ZipFile(zip_files[0]) as z:
    csv_name = z.namelist()[0]
    print(f"File inside zip: {csv_name}")
    with z.open(csv_name) as f:
        df_test = pd.read_csv(f, usecols = columns_needed, nrows=5)
print(f"Shape: {df_test.shape}")
print(f"\nColumns: {df_test.columns.tolist()}")
print(f"\nSample:\n{df_test}")

Found 144 zip files
First: data/raw/flights_2015_01.zip
Last: data/raw/flights_2026_12.zip
File inside zip: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2015_1.csv
Shape: (5, 18)

Columns: ['FlightDate', 'Reporting_Airline', 'Origin', 'OriginStateName', 'Dest', 'DestStateName', 'CRSDepTime', 'DepDelay', 'ArrDelay', 'Cancelled', 'Diverted', 'AirTime', 'Distance', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']

Sample:
   FlightDate Reporting_Airline Origin OriginStateName Dest DestStateName  \
0  2015-01-01                AA    JFK        New York  LAX    California   
1  2015-01-02                AA    JFK        New York  LAX    California   
2  2015-01-03                AA    JFK        New York  LAX    California   
3  2015-01-04                AA    JFK        New York  LAX    California   
4  2015-01-05                AA    JFK        New York  LAX    California   

   CRSDepTime  DepDelay  ArrDelay  Cancelled  Diverted  AirTime  

In [3]:
conn = snowflake.connector.connect(
    user="USER",
    password="PASSWORD",
    account="ACCOUNTID",
    warehouse="FLIGHTS_WH",
    database="FLIGHTS_DB",
    schema="RAW"
)

total_rows = 0
min_file_size = 1 * 1024 * 1024

for zip_path in zip_files:
    file_size = os.path.getsize(zip_path)
    if file_size < min_file_size:
        print(f"Skipping {zip_path} - too small ({file_size/1024:.0f}KB) ")
        continue
    try:
        with zipfile.ZipFile(zip_path) as z:
            csv_name = z.namelist()[0]
            with z.open(csv_name) as f:
                df = pd.read_csv(f, usecols=columns_needed, encoding='latin1')
                
        df.columns = [col.upper() for col in df.columns]

        success, nchunks,nrows, _ = write_pandas(
            conn, df,"FLIGHTS",
            auto_create_table=False
        )

        total_rows += nrows
        print(f"Loaded {zip_path} — {nrows:,} rows (total: {total_rows:,})")
    
    except Exception as e:
        print(f"Error on {zip_path}: {e}")
        
conn.close()
print(f"\nDone! Total rows loaded: {total_rows:,}")

Loaded data/raw/flights_2019_01.zip — 583,985 rows (total: 583,985)
Loaded data/raw/flights_2019_02.zip — 533,175 rows (total: 1,117,160)
Loaded data/raw/flights_2019_03.zip — 632,074 rows (total: 1,749,234)
Loaded data/raw/flights_2019_04.zip — 612,023 rows (total: 2,361,257)
Loaded data/raw/flights_2019_05.zip — 636,390 rows (total: 2,997,647)
Loaded data/raw/flights_2019_06.zip — 636,691 rows (total: 3,634,338)
Loaded data/raw/flights_2019_07.zip — 659,029 rows (total: 4,293,367)
Loaded data/raw/flights_2019_08.zip — 658,461 rows (total: 4,951,828)
Loaded data/raw/flights_2019_09.zip — 605,979 rows (total: 5,557,807)
Loaded data/raw/flights_2019_10.zip — 636,014 rows (total: 6,193,821)
Loaded data/raw/flights_2019_11.zip — 602,453 rows (total: 6,796,274)
Loaded data/raw/flights_2019_12.zip — 625,763 rows (total: 7,422,037)
Loaded data/raw/flights_2020_01.zip — 607,346 rows (total: 8,029,383)
Loaded data/raw/flights_2020_02.zip — 574,268 rows (total: 8,603,651)
Loaded data/raw/flight

In [1]:
import snowflake.connector
import pandas as pd
import os

os.chdir("/Users/adnan/Desktop/us-flight-delays")

conn = snowflake.connector.connect(
    user="USER",
    password="PASSWORD",
    account="ACCOUNTID",
    warehouse="FLIGHTS_WH",
    database="FLIGHTS_DB",
    schema="DBT_DEV"
)

marts = {
    "data/clean/mart_airline_performance.csv": "SELECT * FROM mart_airline_performance",
    "data/clean/mart_airport_performance.csv": "SELECT * FROM mart_airport_performance",
    "data/clean/mart_delay_causes.csv": "SELECT * FROM mart_delay_causes",
    "data/clean/mart_monthly_trends.csv": "SELECT * FROM mart_monthly_trends"
}

for path, query in marts.items():
    df = pd.read_sql(query, conn)
    df.to_csv(path, index=False)
    print(f"Exported {len(df)} rows → {path}")

conn.close()
print("\nDone!")

/var/folders/nb/8t174vjx6lg6v6rvzj0zdjm80000gp/T/ipykernel_5614/2800055900.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Exported 125 rows → data/clean/mart_airline_performance.csv


/var/folders/nb/8t174vjx6lg6v6rvzj0zdjm80000gp/T/ipykernel_5614/2800055900.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Exported 2859 rows → data/clean/mart_airport_performance.csv


/var/folders/nb/8t174vjx6lg6v6rvzj0zdjm80000gp/T/ipykernel_5614/2800055900.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Exported 415 rows → data/clean/mart_delay_causes.csv


/var/folders/nb/8t174vjx6lg6v6rvzj0zdjm80000gp/T/ipykernel_5614/2800055900.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Exported 83 rows → data/clean/mart_monthly_trends.csv

Done!


In [3]:
zip_files_new = [f for f in zip_files if any(f'flights_{year}_' in f for year in [2016, 2017, 2018])]
print(f"Files to load: {len(zip_files_new)}")
for path in zip_files_new:
    print(path)

Files to load: 36
data/raw/flights_2016_01.zip
data/raw/flights_2016_02.zip
data/raw/flights_2016_03.zip
data/raw/flights_2016_04.zip
data/raw/flights_2016_05.zip
data/raw/flights_2016_06.zip
data/raw/flights_2016_07.zip
data/raw/flights_2016_08.zip
data/raw/flights_2016_09.zip
data/raw/flights_2016_10.zip
data/raw/flights_2016_11.zip
data/raw/flights_2016_12.zip
data/raw/flights_2017_01.zip
data/raw/flights_2017_02.zip
data/raw/flights_2017_03.zip
data/raw/flights_2017_04.zip
data/raw/flights_2017_05.zip
data/raw/flights_2017_06.zip
data/raw/flights_2017_07.zip
data/raw/flights_2017_08.zip
data/raw/flights_2017_09.zip
data/raw/flights_2017_10.zip
data/raw/flights_2017_11.zip
data/raw/flights_2017_12.zip
data/raw/flights_2018_01.zip
data/raw/flights_2018_02.zip
data/raw/flights_2018_03.zip
data/raw/flights_2018_04.zip
data/raw/flights_2018_05.zip
data/raw/flights_2018_06.zip
data/raw/flights_2018_07.zip
data/raw/flights_2018_08.zip
data/raw/flights_2018_09.zip
data/raw/flights_2018_10.

In [4]:
conn = snowflake.connector.connect(
    user="USER",
    password="PASSWORD",
    account="ACCOUNTID",
    warehouse="FLIGHTS_WH",
    database="FLIGHTS_DB",
    schema="RAW"
)

total_rows = 0
min_file_size = 1 * 1024 * 1024

for zip_path in zip_files_new:
    file_size = os.path.getsize(zip_path)
    if file_size < min_file_size:
        print(f"Skipping {zip_path} — too small ({file_size/1024:.0f}KB)")
        continue
    
    try:
        with zipfile.ZipFile(zip_path) as z:
            csv_name = z.namelist()[0]
            with z.open(csv_name) as f:
                df = pd.read_csv(f, usecols=columns_needed, encoding='latin1')
        
        df.columns = [col.upper() for col in df.columns]
        
        success, nchunks, nrows, _ = write_pandas(
            conn, df, "FLIGHTS",
            auto_create_table=False
        )
        
        total_rows += nrows
        print(f"Loaded {zip_path} — {nrows:,} rows (total: {total_rows:,})")
    
    except Exception as e:
        print(f"Error on {zip_path}: {e}")

conn.close()
print(f"\nDone! Total rows loaded: {total_rows:,}")

Loaded data/raw/flights_2016_01.zip — 445,827 rows (total: 445,827)
Loaded data/raw/flights_2016_02.zip — 423,889 rows (total: 869,716)
Loaded data/raw/flights_2016_03.zip — 479,122 rows (total: 1,348,838)
Loaded data/raw/flights_2016_04.zip — 461,630 rows (total: 1,810,468)
Loaded data/raw/flights_2016_05.zip — 479,358 rows (total: 2,289,826)
Loaded data/raw/flights_2016_06.zip — 487,637 rows (total: 2,777,463)
Loaded data/raw/flights_2016_07.zip — 502,457 rows (total: 3,279,920)
Loaded data/raw/flights_2016_08.zip — 498,347 rows (total: 3,778,267)
Loaded data/raw/flights_2016_09.zip — 454,878 rows (total: 4,233,145)
Loaded data/raw/flights_2016_10.zip — 472,626 rows (total: 4,705,771)
Loaded data/raw/flights_2016_11.zip — 450,938 rows (total: 5,156,709)
Loaded data/raw/flights_2016_12.zip — 460,949 rows (total: 5,617,658)
Loaded data/raw/flights_2017_01.zip — 450,017 rows (total: 6,067,675)
Loaded data/raw/flights_2017_02.zip — 410,517 rows (total: 6,478,192)
Loaded data/raw/flights_

In [5]:
import pandas as pd
import os

os.chdir("/Users/adnan/Desktop/us-flight-delays")

# Airline lookup table
airline_lookup = {
    'AA': 'American Airlines',
    'DL': 'Delta Air Lines',
    'UA': 'United Airlines',
    'WN': 'Southwest Airlines',
    'B6': 'JetBlue Airways',
    'AS': 'Alaska Airlines',
    'NK': 'Spirit Airlines',
    'F9': 'Frontier Airlines',
    'G4': 'Allegiant Air',
    'HA': 'Hawaiian Airlines',
    'SY': 'Sun Country Airlines',
    'MQ': 'Envoy Air',
    'OO': 'SkyWest Airlines',
    'YX': 'Republic Airways',
    'OH': 'PSA Airlines',
    'YV': 'Mesa Airlines',
    'CP': 'Compass Airlines',
    'EV': 'ExpressJet Airlines',
    'QX': 'Horizon Air',
    'C5': 'CommutAir',
    'ZW': 'Air Wisconsin',
    'PT': 'Piedmont Airlines',
    '9E': 'Endeavor Air',
    'G7': 'GoJet Airlines',
    'VX': 'Virgin America'
}

df_airlines = pd.DataFrame(list(airline_lookup.items()), columns=['CARRIER_CODE', 'AIRLINE_NAME'])
df_airlines.to_csv("data/raw/airline_lookup.csv", index=False)
print(f"Saved airline lookup: {len(df_airlines)} carriers")
print(df_airlines)

Saved airline lookup: 25 carriers
   CARRIER_CODE          AIRLINE_NAME
0            AA     American Airlines
1            DL       Delta Air Lines
2            UA       United Airlines
3            WN    Southwest Airlines
4            B6       JetBlue Airways
5            AS       Alaska Airlines
6            NK       Spirit Airlines
7            F9     Frontier Airlines
8            G4         Allegiant Air
9            HA     Hawaiian Airlines
10           SY  Sun Country Airlines
11           MQ             Envoy Air
12           OO      SkyWest Airlines
13           YX      Republic Airways
14           OH          PSA Airlines
15           YV         Mesa Airlines
16           CP      Compass Airlines
17           EV   ExpressJet Airlines
18           QX           Horizon Air
19           C5             CommutAir
20           ZW         Air Wisconsin
21           PT     Piedmont Airlines
22           9E          Endeavor Air
23           G7        GoJet Airlines
24           VX 

In [6]:
import requests


url = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/airports.dat"
response = requests.get(url)

# Parsing the dat file
import io
cols = ['airport_id', 'name', 'city', 'country', 'iata_code', 'icao_code', 
        'latitude', 'longitude', 'altitude', 'timezone', 'dst', 'tz', 'type', 'source']

df_airports = pd.read_csv(io.StringIO(response.text), header=None, names=cols)

# Filtering to US airports only and keeping relevant columns
df_airports_us = df_airports[
    (df_airports['country'] == 'United States') & 
    (df_airports['iata_code'] != '\\N') &
    (df_airports['iata_code'].str.len() == 3)
][['iata_code', 'name', 'city', 'latitude', 'longitude']].copy()

df_airports_us.columns = ['AIRPORT_CODE', 'AIRPORT_NAME', 'CITY', 'LATITUDE', 'LONGITUDE']
df_airports_us = df_airports_us.reset_index(drop=True)

df_airports_us.to_csv("data/raw/airport_lookup.csv", index=False)
print(f"Saved airport lookup: {len(df_airports_us)} US airports")
print(df_airports_us.head(10))

Saved airport lookup: 1251 US airports
  AIRPORT_CODE                  AIRPORT_NAME              CITY   LATITUDE  \
0          BTI    Barter Island LRRS Airport     Barter Island  70.134003   
1          LUR    Cape Lisburne LRRS Airport     Cape Lisburne  68.875099   
2          PIZ        Point Lay LRRS Airport         Point Lay  69.732903   
3          ITO    Hilo International Airport              Hilo  19.721399   
4          ORL     Orlando Executive Airport           Orlando  28.545500   
5          BTT               Bettles Airport           Bettles  66.913902   
6          UTO  Indian Mountain LRRS Airport  Indian Mountains  65.992798   
7          FYU            Fort Yukon Airport        Fort Yukon  66.571503   
8          SVW       Sparrevohn LRRS Airport        Sparrevohn  61.097401   
9          FRN          Bryant Army Heliport   Fort Richardson  61.266399   

    LONGITUDE  
0 -143.582001  
1 -166.110001  
2 -163.005005  
3 -155.048004  
4  -81.332901  
5 -151.529007  
6

In [10]:
conn = snowflake.connector.connect(
    user="USER",
    password="PASSWORD",
    account="ACCOUNTID",
    warehouse="FLIGHTS_WH",
    database="FLIGHTS_DB",
    schema="RAW"
)

df_airlines = pd.read_csv("data/raw/airline_lookup.csv")
write_pandas(conn, df_airlines, "AIRLINE_LOOKUP")
print(f"Loaded airline lookup: {len(df_airlines)} rows")

df_airports = pd.read_csv("data/raw/airport_lookup.csv")
write_pandas(conn, df_airports, "AIRPORT_LOOKUP")
print(f"Loaded airport lookup: {len(df_airports)} rows")

conn.close()
print("Done!")

Loaded airline lookup: 25 rows
Loaded airport lookup: 1251 rows
Done!


In [11]:
conn = snowflake.connector.connect(
    user="USER",
    password="PASSWORD",
    account="ACCOUNTID",
    warehouse="FLIGHTS_WH",
    database="FLIGHTS_DB",
    schema="DBT_DEV"
)

marts = {
    "data/clean/mart_airline_performance.csv": "SELECT * FROM mart_airline_performance",
    "data/clean/mart_airport_performance.csv": "SELECT * FROM mart_airport_performance",
    "data/clean/mart_delay_causes.csv": "SELECT * FROM mart_delay_causes",
    "data/clean/mart_monthly_trends.csv": "SELECT * FROM mart_monthly_trends"
}

for path, query in marts.items():
    df = pd.read_sql(query, conn)
    df.to_csv(path, index=False)
    print(f"Exported {len(df)} rows → {path}")

conn.close()
print("\nDone!")

/var/folders/nb/8t174vjx6lg6v6rvzj0zdjm80000gp/T/ipykernel_2620/918772874.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Exported 167 rows → data/clean/mart_airline_performance.csv


/var/folders/nb/8t174vjx6lg6v6rvzj0zdjm80000gp/T/ipykernel_2620/918772874.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Exported 3847 rows → data/clean/mart_airport_performance.csv


/var/folders/nb/8t174vjx6lg6v6rvzj0zdjm80000gp/T/ipykernel_2620/918772874.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Exported 595 rows → data/clean/mart_delay_causes.csv


/var/folders/nb/8t174vjx6lg6v6rvzj0zdjm80000gp/T/ipykernel_2620/918772874.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Exported 119 rows → data/clean/mart_monthly_trends.csv

Done!
